# UDA-Hub Core Database Setup

This notebook seeds the UDA-Hub core database (`udahub.db`) with:
- Account record for CultPass
- 18 knowledge base articles across all required categories
- Sample user and ticket data
- Customer-scoped conversation memory records

In [1]:
import json
import uuid
from datetime import datetime, timezone
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

C:\Users\eduardo.nicacio\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## Initialize Database

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

2026-07-23 19:57:59,341 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-07-23 19:57:59,342 INFO sqlalchemy.engine.Engine COMMIT


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

## Seed Account

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Load Knowledge Base Articles

The `cultpass_articles.jsonl` file contains 18 articles covering all required categories:
billing, account, technical, subscription, reservation, content, onboarding.

In [7]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [8]:
print(f"Loaded {len(cultpass_articles)} articles")
for i, article in enumerate(cultpass_articles):
    print(f"  {i+1}. {article['title']} [{article['tags']}]")

Loaded 18 articles
  1. How to Reserve a Spot for an Event [reservation, events, booking, attendance]
  2. What's Included in a CultPass Subscription [subscription, benefits, pricing, access]
  3. How to Cancel or Pause a Subscription [cancelation, pause, subscription, billing]
  4. How to Handle Login Issues? [login, password, access, escalation]
  5. How to Request a Refund [refund, billing, payment, money back]
  6. Understanding Your Billing Cycle [billing, cycle, payment, charges, subscription]
  7. How to Dispute a Double Charge [double charge, duplicate, billing, refund, dispute]
  8. How to Update Your Email Address [email, profile, account, update]
  9. How to Delete Your CultPass Account [delete, account removal, data, permanent]
  10. App Crashes on Event Reservation [crash, app issue, technical, reservation, bug]
  11. QR Code Not Scanning at Check-In [qr code, scanning, check-in, event, technical]
  12. Video Streaming Issues on Web [streaming, video, buffering, technical,

In [9]:
if len(cultpass_articles) < 14:
    raise AssertionError(f"Expected at least 14 articles, got {len(cultpass_articles)}")
print(f"Article count check passed: {len(cultpass_articles)} >= 14")

Article count check passed: 18 >= 14


In [10]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb)
    print(f"Inserted {len(kb)} knowledge articles")

Inserted 18 knowledge articles


## Seed User and Ticket

In [11]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [12]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [13]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )

    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])
    seeded_ticket_id = ticket.ticket_id
    print(f"Seeded user {user.user_name} (external_id={user.external_user_id})")
    print(f"Seeded ticket {ticket.ticket_id}")

Seeded user Alice Kingsley (external_id=a4ab87)
Seeded ticket b68a187a-f14a-4bb6-916b-8fa2114d3d38


## Seed Conversation Memory (Customer-Scoped)

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    existing_ticket = user.tickets[0] if user.tickets else None

    if existing_ticket:
        memory = udahub.ConversationMemory(
            memory_id=str(uuid.uuid4()),
            account_id=account_id,
            customer_id=ticket_info["owner_id"],
            ticket_id=existing_ticket.ticket_id,
            summary="Customer had login issues. Resolved by guiding through password reset process.",
            category="account",
            resolution_type="resolved",
            created_at=datetime.now(timezone.utc),
        )
        session.add(memory)
        print(f"Seeded memory record for customer_id={ticket_info['owner_id']}")

Seeded memory record for customer_id=a4ab87


## Verification Tests

In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(f"Account: {account}")

Account: <Account(account_id='cultpass', account_name='CultPass Card')>


In [16]:
with get_session(engine) as session:
    articles = session.query(udahub.Knowledge).all()
    print(f"Knowledge articles in DB: {len(articles)}")
    for article in articles:
        print(f"  - {article.title} [{article.tags}]")
    assert len(articles) >= 14, f"Expected >= 14 articles, got {len(articles)}"
    print(f"\nArticle count verified: {len(articles)} >= 14")

Knowledge articles in DB: 18
  - How to Reserve a Spot for an Event [reservation, events, booking, attendance]
  - What's Included in a CultPass Subscription [subscription, benefits, pricing, access]
  - How to Cancel or Pause a Subscription [cancelation, pause, subscription, billing]
  - How to Handle Login Issues? [login, password, access, escalation]
  - How to Request a Refund [refund, billing, payment, money back]
  - Understanding Your Billing Cycle [billing, cycle, payment, charges, subscription]
  - How to Dispute a Double Charge [double charge, duplicate, billing, refund, dispute]
  - How to Update Your Email Address [email, profile, account, update]
  - How to Delete Your CultPass Account [delete, account removal, data, permanent]
  - App Crashes on Event Reservation [crash, app issue, technical, reservation, bug]
  - QR Code Not Scanning at Check-In [qr code, scanning, check-in, event, technical]
  - Video Streaming Issues on Web [streaming, video, buffering, technical, play

In [17]:
from agentic.tools.kb_search_tool import kb_search

test_queries = [
    ("How do I reserve an event?", "reservation"),
    ("I was charged twice", "billing"),
    ("The app crashes", "technical"),
    ("I want to upgrade my plan", "subscription"),
    ("How do I get started?", "onboarding"),
]

all_passed = True
for query, expected_category in test_queries:
    results = kb_search(query, category=expected_category, account_id=account_id)
    if results:
        print(f"  PASS: '{query}' -> {results[0]['title']} (score={results[0]['score']})")
    else:
        print(f"  FAIL: '{query}' -> no results for category '{expected_category}'")
        all_passed = False

assert all_passed, "Some KB retrieval tests failed"
print("\nAll KB retrieval tests passed!")

  PASS: 'How do I reserve an event?' -> How to Reserve a Spot for an Event (score=22)
  PASS: 'I was charged twice' -> How to Dispute a Double Charge (score=21)
  PASS: 'The app crashes' -> App Crashes on Event Reservation (score=36)
  PASS: 'I want to upgrade my plan' -> Upgrading from Basic to Premium (score=16)
  PASS: 'How do I get started?' -> Getting Started with CultPass (score=30)

All KB retrieval tests passed!


In [18]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='a5caeac8-0b42-4f29-a404-20a2e742c974', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [19]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='2ddf58e2-82bd-4149-8d9b-ee4eb48ee351', role='user', content='I can't log in to my Cultpass ...')>


In [20]:
with get_session(engine) as session:
    memories = session.query(udahub.ConversationMemory).all()
    print(f"ConversationMemory records: {len(memories)}")
    for m in memories:
        print(f"  customer_id={m.customer_id}, category={m.category}, resolution_type={m.resolution_type}")
        print(f"    summary: {m.summary}")

ConversationMemory records: 1
  customer_id=a4ab87, category=account, resolution_type=resolved
    summary: Customer had login issues. Resolved by guiding through password reset process.


In [21]:
with get_session(engine) as session:
    customer_memories = session.query(udahub.ConversationMemory).filter(
        udahub.ConversationMemory.customer_id == ticket_info["owner_id"]
    ).all()
    print(f"Memories for customer '{ticket_info['owner_id']}': {len(customer_memories)}")
    assert len(customer_memories) >= 1, "Expected at least 1 customer-scoped memory"
    print("Customer-scoped memory test passed!")

Memories for customer 'a4ab87': 1
Customer-scoped memory test passed!
